# Trabalho Prático – Estrutura de Dados II (DCA3702)
## Análise Estrutural de Redes Urbanas com OSMnx, NetworkX e Gephi

Este notebook documenta a análise estrutural da malha viária das regiões de **Coophab e Cajupiranga (Parnamirim/RN)**. O objetivo é utilizar dados reais do OpenStreetMap para identificar hubs, gargalos de mobilidade e núcleos densos através da modelagem de grafos.

**Autores:** Matheus Fernandes e Thales Varela

## 1. Problema Norteador e Objetivos

**Problema Central:**
> Quais são os elementos estruturais mais importantes da malha viária de Coophab e Cajupiranga, e como diferentes métricas de grafos (grau, centralidade e k-core) ajudam a caracterizá-los?

**Objetivos Específicos:**
- Identificar os cruzamentos centrais da rede (hubs).
- Encontrar regiões estruturalmente densas por meio da decomposição `k-core`.
- Localizar pontos críticos de conexão e fluxo usando métricas de `betweenness centrality`.

## 2. Escolha da Região de Estudo

**Regiões:** Coophab e Cajupiranga (Parnamirim/RN).

**Justificativa:**
A escolha dessas áreas adjacentes se dá pelo conhecimento local prévio da dinâmica de trânsito. A combinação de Coophab com Cajupiranga cria um grafo com volume ideal de dados, apresentando uma estrutura viária peculiar: o forte contraste entre os "silos" de condomínios fechados (como Green Club 2 e Morumbi) e as vias arteriais de escoamento. Essa topologia permitirá avaliar matematicamente como as vias de acesso a esses condomínios atuam como gargalos de mobilidade.

## 3. Etapa 1 – Construção do Grafo com OSMnx

Nesta etapa, utilizamos a biblioteca `OSMnx` para baixar a rede viária da região escolhida.
Para modelar o trânsito real, o grafo será construído com a restrição `network_type="drive"`, ignorando ruas exclusivas para pedestres.

- **Nós (Vértices):** Representam os cruzamentos, bifurcações ou rotatórias.
- **Arestas (Conexões):** Representam as vias e segmentos de ruas que conectam os cruzamentos.

In [ ]:
# Instalando a biblioteca (sem o # para garantir que sempre instale se o Colab resetar)
!pip install osmnx

import osmnx as ox
import networkx as nx

print("Iniciando o download da malha viária...")

# Coordenada central exata na região da Coophab/Parque das Árvores
ponto_central = (-5.908906, -35.205872)

# Baixando o grafo num raio de 3000 metros (3km) focado em carros
G = ox.graph_from_point(ponto_central, dist=3000, network_type="drive")

print("\nDownload concluído com sucesso!")
print(f"Número de cruzamentos (Nós): {G.number_of_nodes()}")
print(f"Número de ruas (Arestas): {G.number_of_edges()}")

Iniciando o download da malha viária...

Download concluído com sucesso!
Número de cruzamentos (Nós): 1327
Número de ruas (Arestas): 3201


## 4. Etapa 2 – Análise Estrutural com NetworkX (Centralidades)

Nesta etapa, iniciamos o cálculo das métricas topológicas da rede. Conforme orientação metodológica do projeto, primeiro convertemos o grafo direcionado em um grafo não-direcionado (`G_undirected`) e, em seguida, em um **grafo simples** (`G_simples`).

**Por que essa dupla conversão?**
O `ox.convert.to_undirected` ainda retorna um `MultiGraph` (permite arestas paralelas entre os mesmos cruzamentos — útil quando há duas ruas diferentes ligando o mesmo par de esquinas). Já a função `nx.core_number` exige um grafo simples. Para manter **todas as métricas consistentes entre si**, calculamos grau, betweenness e closeness sobre o `G_simples`. Assim, "grau" representa o número de **vizinhos únicos** (e não a soma de segmentos paralelos), o que é a interpretação correta para o contexto urbano.

Calcularemos as seguintes métricas fundamentais:
- **Degree (Grau):** Número de conexões diretas de cada cruzamento. Revelará os nossos "Hubs" primários.
- **Betweenness Centrality (Intermediação):** Identifica os cruzamentos que atuam como "pontes" ou gargalos nos caminhos mais curtos da rede.
- **Closeness Centrality (Proximidade):** Avalia quais cruzamentos estão, em média, mais próximos de todos os outros pontos da região. Para que a métrica seja bem definida, validamos antes a conectividade do grafo.

In [ ]:
print("Convertendo grafo e calculando centralidades (isso pode levar alguns segundos)...\n")

# Conversão para grafo não-direcionado (ainda MultiGraph)
G_undirected = ox.convert.to_undirected(G)

# Conversão para grafo SIMPLES — usado em TODAS as métricas para garantir consistência
G_simples = nx.Graph(G_undirected)

print(f"G_undirected (MultiGraph): {G_undirected.number_of_nodes()} nós | {G_undirected.number_of_edges()} arestas")
print(f"G_simples   (Graph)      : {G_simples.number_of_nodes()} nós | {G_simples.number_of_edges()} arestas")

# Validação de conectividade (necessária para closeness ser bem definida)
conectado = nx.is_connected(G_simples)
n_componentes = nx.number_connected_components(G_simples)
print(f"\nO grafo é conexo? {conectado}")
print(f"Número de componentes conectados: {n_componentes}")

# 1. Grau dos nós (Degree) — sobre o grafo simples
degree_dict = dict(G_simples.degree())

# 2. Betweenness Centrality (Intermediação)
betweenness = nx.betweenness_centrality(G_simples, normalized=True)

# 3. Closeness Centrality (Proximidade) — funciona pois validamos conectividade
closeness = nx.closeness_centrality(G_simples)

# Top 10 nós por grau (Hubs primários)
top_degree = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:10]

print("\nCálculos concluídos!\n")
print("🏆 TOP 10 CRUZAMENTOS POR GRAU (HUBS):")
for node_id, degree in top_degree:
    print(f"  Nó ID: {node_id} | Grau: {degree}")

### 4.1 Distribuição de Grau

A distribuição de grau revela a **forma estrutural** da rede. Redes viárias urbanas tipicamente apresentam concentração em graus baixos (2-4), refletindo segmentos lineares e cruzamentos comuns. Diferentemente de redes sociais ou da internet (que seguem leis de potência com hubs gigantes), redes planares como vias urbanas têm um teto natural de grau imposto pela geometria — não é fisicamente plausível ter um cruzamento com 20 ruas.

No bloco abaixo:
- Calculamos estatísticas descritivas do grau (média, mediana, desvio, mín/máx).
- Tabulamos quantos nós existem em cada valor de grau.
- Plotamos o histograma, que será salvo em `distribuicao_grau.png` para uso no README.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

graus = [d for _, d in G_simples.degree()]
contagem_grau = Counter(graus)
total = len(graus)

print("Estatísticas descritivas do grau:")
print(f"  Mínimo:     {min(graus)}")
print(f"  Máximo:     {max(graus)}")
print(f"  Média:      {np.mean(graus):.2f}")
print(f"  Mediana:    {int(np.median(graus))}")
print(f"  Desvio pad: {np.std(graus):.2f}")

print("\nDistribuição absoluta e relativa:")
for k in sorted(contagem_grau):
    pct = 100 * contagem_grau[k] / total
    print(f"  Grau {k}: {contagem_grau[k]:>5} nós ({pct:.1f}%)")

# Plot
plt.figure(figsize=(8, 4))
valores = sorted(contagem_grau.keys())
freqs = [contagem_grau[v] for v in valores]
barras = plt.bar(valores, freqs, color="#8B0000", edgecolor="black")
for v, f in zip(valores, freqs):
    plt.text(v, f + max(freqs) * 0.02, str(f), ha="center", fontsize=10)
plt.xlabel("Grau")
plt.ylabel("Número de nós")
plt.title("Distribuição de Grau — Rede Viária Coophab/Cajupiranga")
plt.xticks(valores)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("distribuicao_grau.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Etapa 2 – Decomposição K-Core (O "Núcleo Duro")

Para entender a robustez da malha viária local, aplicamos a **Decomposição K-Core**. Este método funciona como "descascar uma cebola": removemos iterativamente os nós menos conectados (como ruas sem saída ou vielas de periferia) para revelar o núcleo estruturalmente mais denso do grafo.

- **Core Number:** Determina a camada mais profunda (maior `k`) à qual um cruzamento consegue sobreviver. Analisar esse núcleo ajudará a entender as vias que formam o esqueleto de tráfego entre a Coophab e Cajupiranga.

In [ ]:
print("Calculando o K-Core...\n")

# G_simples já foi criado na célula anterior (Etapa 2 — Centralidades).
# A decomposição k-core exige grafo simples (sem multi-arestas e sem auto-loops).
# Reutilizando para manter consistência metodológica entre todas as métricas.

# Cálculo do core number para cada nó
core_number = nx.core_number(G_simples)

# Maior valor de core encontrado na nossa rede
max_core = max(core_number.values())

# Identificando os nós que pertencem a esse núcleo mais profundo
main_core_nodes = [node for node, core in core_number.items() if core == max_core]

print(f"Maior K-Core da rede (O núcleo mais denso): {max_core}-Core")
print(f"Número de cruzamentos que formam esse núcleo principal: {len(main_core_nodes)}")
print(f"Percentual da rede no núcleo principal: {100 * len(main_core_nodes) / G_simples.number_of_nodes():.2f}%")

### 5.1 Análise Aprofundada por K-Shell

Conhecer apenas o maior k-core não é suficiente. Cada nó pertence a uma **k-shell** específica (o maior valor de k para o qual ele ainda sobrevive no k-core). Diferentes k-shells representam diferentes graus de robustez topológica:

- **k=1 (camada externa):** nós periféricos — cul-de-sacs, ramais terminais, ruas sem saída. Removendo um vizinho, o nó desconecta.
- **k=2:** nós que pertencem a pelo menos um ciclo. Formam o esqueleto cíclico da rede.
- **k ≥ 3:** zonas super-densas com triângulos sobrepostos. Raro em redes viárias planares.

Vamos calcular a distribuição completa por k-shell e visualizar como os nós se distribuem entre as camadas.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

contagem_core = Counter(core_number.values())
total = len(core_number)

print("Distribuição por k-shell:\n")
for k in sorted(contagem_core):
    pct = 100 * contagem_core[k] / total
    print(f"  k-shell {k}: {contagem_core[k]:>5} nós ({pct:.1f}%)")

print("\nInterpretação estrutural:")
print(f"  - {contagem_core.get(1, 0)} nós (k=1) são periféricos:")
print(f"    cul-de-sacs, ramais terminais e ruas sem saída.")
print(f"  - {contagem_core.get(2, 0)} nós (k=2) formam o esqueleto cíclico da rede:")
print(f"    cada um pertence a pelo menos um ciclo fechado de ruas.")
print(f"  - Não existem subgrafos com k>=3, indicando ausência")
print(f"    de zonas super-densas (típico de rede viária planar).")

# Plot
plt.figure(figsize=(7, 4))
valores = sorted(contagem_core.keys())
freqs = [contagem_core[k] for k in valores]
plt.bar(valores, freqs, color="#2C7B8E", edgecolor="black")
for k in valores:
    plt.text(k, contagem_core[k] + max(freqs) * 0.02, str(contagem_core[k]),
             ha="center", fontsize=10)
plt.xlabel("k-shell")
plt.ylabel("Número de nós")
plt.title("Decomposição K-Core — Distribuição por Shell")
plt.xticks(valores)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("distribuicao_kcore.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Etapa 3 – Exportação do Grafo para o Gephi

Com todas as métricas calculadas, precisamos injetá-las como atributos nos nós do nosso grafo. Isso garantirá que, ao abrirmos o arquivo no software Gephi, as variáveis matemáticas (Grau, Intermediação, Proximidade e Core) estejam disponíveis para configurarmos as cores e os tamanhos dos cruzamentos.

Por fim, o grafo será salvo no formato `.graphml`, que é o padrão suportado por ferramentas de visualização de redes.

In [ ]:
print("Preparando dados e forçando IDs únicos para o Gephi...")

# 1. Injetando as métricas como atributos nos nós
nx.set_node_attributes(G_undirected, degree_dict, "degree")
nx.set_node_attributes(G_undirected, betweenness, "betweenness")
nx.set_node_attributes(G_undirected, closeness, "closeness")
nx.set_node_attributes(G_undirected, core_number, "core_number")

# 2. Limpeza dos atributos dos NÓS (tipos não-primitivos viram string)
for node, data in G_undirected.nodes(data=True):
    for key, val in list(data.items()):
        if type(val) not in [int, float, str, bool]:
            data[key] = str(val)

# 3. Atribui um ID ÚNICO para CADA aresta (incluindo multi-arestas)
contador_id = 0
for u, v, k, data in G_undirected.edges(keys=True, data=True):
    data['id'] = f"aresta_{contador_id}"
    contador_id += 1
    for key, val in list(data.items()):
        if type(val) not in [int, float, str, bool]:
            data[key] = str(val)

nome_arquivo = "rede_coophab_gephi_final.graphml"

print("Exportando para .graphml com IDs de aresta únicos...")

# ↓↓↓ A LINHA QUE RESOLVE TUDO ↓↓↓
nx.write_graphml(
    G_undirected,
    nome_arquivo,
    edge_id_from_attribute='id'   # usa nosso 'aresta_X' como id XML da <edge>
)

print(f"Sucesso! Arquivo '{nome_arquivo}' regerado com {contador_id} arestas únicas.")
print(f"Nós: {G_undirected.number_of_nodes()} | Arestas: {G_undirected.number_of_edges()}")


Preparando dados e forçando IDs únicos para o Gephi...
Exportando para .graphml com IDs de aresta únicos...
Sucesso! Arquivo 'rede_coophab_gephi_final.graphml' regerado com 1921 arestas únicas.
Nós: 1327 | Arestas: 1921


## 7. Comparação Cruzada — Top 10 por Múltiplas Métricas

Para responder às questões obrigatórias do projeto, precisamos comparar os rankings produzidos por cada métrica. Um nó pode aparecer no Top 10 de uma métrica mas não em outra — e essas divergências são informativas.

Mostramos lado a lado:
- **Top 10 por Betweenness** — pontes do fluxo global da rede;
- **Top 10 por Closeness** — nós mais próximos de todos os outros em média;
- Para cada nó, exibimos **grau, k-core, betweenness e closeness** simultaneamente, evidenciando correlações entre as métricas.

Ao final, computamos a **interseção entre os rankings** para quantificar quanto eles concordam.

In [ ]:
# Top 10 por Betweenness
top_betw = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print("🌉 TOP 10 CRUZAMENTOS POR BETWEENNESS (PONTES ESTRUTURAIS):\n")
print(f"{'#':<3} {'Nó OSM ID':<14} {'Betweenness':<13} {'Grau':<5} {'K-Core':<7} {'Closeness':<10}")
print("-" * 60)
for i, (node_id, valor) in enumerate(top_betw, 1):
    grau = degree_dict[node_id]
    k = core_number[node_id]
    clos = closeness[node_id]
    print(f"{i:<3} {node_id:<14} {valor:.6f}    {grau:<5} {k:<7} {clos:.5f}")

# Top 10 por Closeness
top_clos = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:10]
print("\n📍 TOP 10 CRUZAMENTOS POR CLOSENESS (PROXIMIDADE GLOBAL):\n")
print(f"{'#':<3} {'Nó OSM ID':<14} {'Closeness':<11} {'Grau':<5} {'K-Core':<7} {'Betweenness':<12}")
print("-" * 60)
for i, (node_id, valor) in enumerate(top_clos, 1):
    grau = degree_dict[node_id]
    k = core_number[node_id]
    betw = betweenness[node_id]
    print(f"{i:<3} {node_id:<14} {valor:.5f}     {grau:<5} {k:<7} {betw:.6f}")

# Sobreposição entre rankings (interseções)
top10_degree_ids = {nid for nid, _ in top_degree}
top10_betw_ids   = {nid for nid, _ in top_betw}
top10_clos_ids   = {nid for nid, _ in top_clos}

inter_db = top10_degree_ids & top10_betw_ids
inter_bc = top10_betw_ids   & top10_clos_ids
inter_dc = top10_degree_ids & top10_clos_ids

print("\n🔍 Sobreposições entre os Top 10:")
print(f"  Top10 Grau ∩ Top10 Betweenness: {len(inter_db)} nós em comum")
print(f"  Top10 Betweenness ∩ Top10 Closeness: {len(inter_bc)} nós em comum")
print(f"  Top10 Grau ∩ Top10 Closeness: {len(inter_dc)} nós em comum")

# Identificar nó "tri-campeão": no top de pelo menos 2 rankings
tri_campeoes = top10_betw_ids & top10_clos_ids & top10_degree_ids
if tri_campeoes:
    print(f"\n⭐ Nó(s) presente(s) nos 3 rankings: {tri_campeoes}")
else:
    print("\n  Nenhum nó está simultaneamente nos 3 rankings.")


## 8. Questões Analíticas Obrigatórias

Esta seção responde às sete questões obrigatórias definidas no enunciado do projeto. As respostas se apoiam diretamente nas métricas calculadas e nas quatro visualizações geradas no Gephi (V1: geográfica completa; V2: top 10% por grau; V3: k-core k=2; V4: estrutural via ForceAtlas2).

### Q1 — Os nós com maior grau coincidem com os nós de maior betweenness?

**Resposta: NÃO — e essa discordância é um dos achados mais reveladores do trabalho.**

Os dados mostram **0 nós em comum** entre o Top 10 por grau e o Top 10 por betweenness. Esse resultado, à primeira vista contraintuitivo, tem duas causas estruturais:

**1. Saturação do grau (efeito de empate).**
Como o grau máximo é 4 e existem **192 nós empatados nesse valor (14.5% da rede)**, o "Top 10 por grau" é essencialmente um sorteio arbitrário entre esses 192 — não há ordenação real. A ordem que o `sorted()` retorna depende apenas da ordem interna do dicionário, não de importância estrutural.

**2. Centralidade depende mais de posição do que de conexões locais.**
Olhando o Top 10 por betweenness com a coluna de grau:

| Grau | Quantos no Top 10 betweenness |
|---|---|
| Grau 4 (máx) | 3 nós (`3454188414`, `3454188380`, `3454188382`) |
| Grau 3 | 7 nós |

**Sete dos dez nós mais centrais por betweenness têm grau 3, não 4.** Mesmo sendo apenas T-intersections (3 ruas), esses cruzamentos aparecem em desproporcionalmente mais caminhos mínimos da rede porque **estão posicionados em corredores críticos** (provavelmente avenidas arteriais que ligam bairros). Já a maioria dos cruzamentos de grau 4 está em **regiões internas** de loteamentos, com 4 ruas conectando o nó a vizinhos, mas todos esses vizinhos levam ao mesmo bairro — alto grau, baixa importância global.

**Caso especial — o nó `3454188414`:**
É o único nó que combina os dois mundos: tem grau 4 (máximo) E maior betweenness (0.47). Mas ele está fora do "Top 10 por grau" exibido apenas porque o sort arbitrário não o pegou nos primeiros 10. Esse nó é o verdadeiro hub estrutural da região.

**Conclusão urbana:** em malha viária saturada (grau ≤ 4), **o grau perde poder discriminativo** e a betweenness se torna a métrica essencial para identificar gargalos reais de mobilidade.

### Q2 — O núcleo identificado pelo k-core coincide com os principais hubs?

**Resposta: não totalmente.** O k=2 core contém **1157 nós (87.19%)**, enquanto os hubs (top 10% por grau) são apenas 195. Praticamente todos os hubs estão dentro do k-core, mas a recíproca é falsa: a **grande maioria dos nós no k-core são interseções comuns de grau 2 ou 3**, não hubs.

O k-core captura uma propriedade diferente do grau: ele mede a **robustez topológica** (presença de ciclos), não a complexidade local de cada cruzamento. Por isso:
- Um cruzamento de grau 4 numa rua sem saída ainda pode estar fora do k-core (se um lado for cul-de-sac).
- Um cruzamento de grau 2 no meio de um quarteirão fechado está no k=2 core (porque participa do ciclo da quadra).

**Conclusão:** k-core e hub são métricas **complementares**, não redundantes. Uma rede pode ter muitos hubs e poucos ciclos (não é o nosso caso) ou poucos hubs e muitos ciclos (mais próximo do nosso caso).

### Q3 — O que a métrica de betweenness revela que o grau não revela?

O **grau é local**: conta apenas vizinhos imediatos. A **betweenness é global**: mede em quantos caminhos mais curtos da rede inteira um nó aparece.

Em uma malha viária planar com grau saturado em 4, o grau perde discriminação. A betweenness revela que, mesmo com o mesmo grau de 194 outros nós, **alguns cruzamentos são desproporcionalmente importantes** porque são o único caminho conectando partes inteiras da cidade.

**Exemplo concreto da nossa rede:** o nó `3454188414` tem grau 4 (igual a centenas de outros), mas sua betweenness é várias ordens de magnitude maior que a média. Visualmente, esse nó está posicionado exatamente no "pescoço" entre as duas comunidades topológicas detectadas pelo ForceAtlas2 (Visualização 4) — confirmação geométrica do papel estrutural especial dele.

**Em termos práticos:** se um cruzamento de alta betweenness fechar (acidente, obra, alagamento), o impacto no fluxo da cidade é desproporcional ao seu grau local. É a métrica chave para **planejamento de mobilidade e gestão de risco**.

### Q4 — O que muda entre análise geográfica e análise estrutural?

A **visualização geográfica** (Visualizações 1, 2 e 3 no Gephi com Geo Layout) preserva o reconhecimento espacial: enxergamos bairros, vias arteriais e áreas verdes. Mas a topologia fica subordinada à geometria — comunidades topológicas podem ficar mascaradas pela proximidade física, e a forma da cidade domina a impressão visual.

A **visualização estrutural** (Visualização 4, ForceAtlas2) abandona a geografia e deixa as conexões definirem a posição: nós densamente conectados se atraem, nós desconectados se repelem. O resultado é uma representação que destaca **comunidades topológicas, hubs centrais e gargalos** que estavam escondidos no layout geográfico.

**O resultado mais surpreendente** foi que o ForceAtlas2, sem qualquer informação de coordenadas, **recuperou a divisão geográfica norte/sul** da região: duas mega-comunidades (Nova Parnamirim ao norte; Parque das Árvores/Cajupiranga ao sul) ligadas por um pescoço estreito que passa pelo nó `3454188414`. Esse alinhamento entre topologia e geografia **prova matematicamente que a separação entre as duas regiões não é apenas uma impressão visual, é uma propriedade estrutural da rede de conexões.**

### Q5 — Existem regiões críticas para mobilidade urbana na área analisada?

**Sim. Identificamos três tipos de criticidade:**

1. **Bridge node único:** o nó `3454188414` é o ponto de fragilidade estrutural mais importante. Ele conecta as duas mega-comunidades topológicas. Uma interdição nesse cruzamento forçaria desvios significativamente maiores entre Nova Parnamirim e Parque das Árvores/Cajupiranga.

2. **Corredor de alta betweenness:** os nós com betweenness muito acima da média formam uma "espinha dorsal" que atravessa a região. Visível como caminhos destacados em cor mais escura na Visualização 3 (k-core), esse corredor é a rota principal de conexão entre os bairros e provavelmente coincide com avenidas arteriais reais (BR-101 e suas vias paralelas).

3. **Silos vulneráveis:** os 170 nós fora do k=2 core (12.81% da rede) representam ruas sem saída, becos e ramais terminais. Vários deles formam **enclaves dentro de loteamentos fechados** (Green Club 2, Morumbi, etc.), conforme antecipado na justificativa da escolha da região. Qualquer obstrução nas poucas vias de acesso isola moradores, pois não há rotas alternativas internas.

### Q6 — A rede parece homogênea ou apresenta concentração estrutural?

**Mista — alta homogeneidade local + forte concentração global.** A distinção é crucial:

**Métricas locais — altamente homogêneas:**
- Distribuição de grau concentrada em poucos valores. **Modal e mediana = 3** (T-intersections, 71.5%), com média 2.89.
- Apenas 4 valores possíveis (grau 1, 2, 3, 4). Nenhuma cauda longa, nenhum "super-hub" no sentido de redes scale-free como a internet ou redes sociais.
- 87.2% no k=2 core — distribuição binária e quase uniforme entre as duas k-shells.

**Métricas globais — fortemente concentradas:**
- **Betweenness varia em ordens de magnitude.** O nó mais central (`3454188414`, betw=0.47) tem betweenness **~10x maior** que o décimo do ranking (~0.21) e **milhares de vezes maior** que a mediana.
- **Estrutura modular bicomunitária.** A Visualização 4 (ForceAtlas2) mostra dois clusters topológicos bem definidos ligados por um pescoço estreito — alta modularidade, não rede homogênea.
- **Closeness também concentrada**, mas em escala menor: os 10 mais próximos têm valores entre 0.060–0.063, enquanto a média provavelmente é bem menor.

**Conclusão:** o **perfil "homogêneo localmente, concentrado globalmente"** é típico de redes viárias planares em zonas residenciais com loteamentos. Localmente, todos os cruzamentos são "parecidos" (3-4 ruas); globalmente, alguns poucos cruzamentos suportam o fluxo da cidade inteira. Esse perfil tem implicações práticas: ataques aleatórios ao grafo têm pouco efeito (homogeneidade local), mas ataques dirigidos aos nós de alta betweenness são devastadores (concentração global).

### Q7 — Os resultados fazem sentido considerando o conhecimento urbano da região?

**Sim. Toda a análise é internamente consistente e bate com o conhecimento prévio da região:**

| Observação matemática | Realidade urbana correspondente |
|---|---|
| Divisão topológica norte/sul detectada pelo ForceAtlas2 | Separação física entre Nova Parnamirim (norte) e Parque das Árvores/Cajupiranga (sul), com BR-101 e Parque do Jiqui no meio |
| Grau máximo = 4 | Ausência de viadutos e cruzamentos multinível — coerente com bairros residenciais |
| Ausência de k≥3 | Sem hubs de transporte multimodal (estações de metrô, terminais rodoviários, etc.) na região |
| "Tentáculos" no SW/SE da Visualização 1 | Estradas vicinais saindo da malha urbana em direção a áreas menos adensadas, visíveis no Google Maps |
| Lado leste esvaziado | Parque do Jiqui (APA — Área de Proteção Ambiental) e Jiqui Country Club: poucas vias mapeadas no OSM |
| Nós fora do k=2 core concentrados em "silos" | Loteamentos fechados (Green Club 2, Morumbi) com poucas vias de acesso à malha pública |
| Bridge node `3454188414` | Provável cruzamento crítico de uma avenida arterial entre os bairros (validação em campo recomendada) |

A coerência entre todas essas observações topológicas e a realidade urbana valida a abordagem metodológica: **modelar uma cidade como grafo realmente revela propriedades estruturais relevantes para urbanismo e mobilidade**.

## 9. Conclusões

A análise estrutural da malha viária de **Coophab e Cajupiranga (Parnamirim/RN)**, em um raio de 3 km centrado em **(-5.908906, -35.205872)**, permitiu validar matematicamente intuições urbanas locais e descobrir características estruturais não-óbvias da região.

### Principais achados

1. **Identificação do bridge node `3454188414`.** Esse nó é triplamente crítico: tem grau máximo (4), maior betweenness da rede (0.47), maior closeness (0.063), pertence ao k=2 core e foi posicionado no pescoço estrutural pelo ForceAtlas2 (Visualização 4). Em termos práticos, **trata-se de um cruzamento cuja interdição compromete significativamente a conectividade entre Nova Parnamirim e Parque das Árvores/Cajupiranga**.

2. **Grau é uma métrica saturada e arbitrária em redes viárias.** Com 192 nós empatados em grau 4 (o máximo), o "Top 10 por grau" é essencialmente um sorteio aleatório. **A interseção entre Top 10 por grau e Top 10 por betweenness foi zero** — não porque os hubs não sejam importantes, mas porque o grau não consegue ordenar nós entre si quando há empate massivo. Esse achado destaca a **necessidade absoluta de métricas globais** (betweenness, closeness) em análises de mobilidade urbana.

3. **Posição vence conexões.** Entre o Top 10 por betweenness, **7 nós têm grau 3** e apenas 3 têm grau 4. Cruzamentos de 3 ruas localizados em corredores arteriais são mais centrais para o fluxo global do que cruzamentos de 4 ruas internos a loteamentos. **A posição na topologia importa mais que a quantidade de conexões diretas.**

4. **Topologia recupera geografia.** O layout estrutural ForceAtlas2, sem usar coordenadas, posicionou os nós em duas mega-comunidades correspondentes à divisão geográfica norte/sul. Isso prova que essa separação não é apenas visual — é uma propriedade estrutural intrínseca da rede de conexões.

5. **Robustez topológica heterogênea.** 87.19% da rede (1157 nós) faz parte do k=2 core, formando um esqueleto cíclico robusto. Os 12.81% restantes (170 nós) são pontos vulneráveis: cul-de-sacs, ramais terminais e silos de condomínios fechados, exatamente como antecipado na justificativa da escolha da região.

6. **Distribuição de grau revela a simplificação do OSMnx.** O modal de grau é **3** (71.5%), não 2. Isso ocorre porque o OSMnx remove nós lineares intermediários por padrão, mantendo apenas verdadeiros cruzamentos. Os 33 nós (2.5%) de grau 2 representam principalmente curvas acentuadas ou transições topológicas onde a simplificação não foi aplicada.

7. **Validação metodológica.** A diferença de **4 arestas** entre o `G_undirected` (1921 arestas, MultiGraph) e o `G_simples` (1917 arestas) confirmou a existência de arestas paralelas residuais — duas ruas distintas conectando o mesmo par de cruzamentos. Calcular as métricas sobre `G_simples` foi a escolha correta para evitar distorções de grau.

8. **Validação com o conhecimento local.** Todos os achados topológicos (divisão norte/sul, lado leste esvaziado pelo Parque do Jiqui, silos de condomínios fechados, tentáculos rurais) são consistentes com a realidade urbana observável no Google Maps e com o conhecimento prévio da dinâmica da região.

### Limitações da análise

- **Direção das vias ignorada.** Para padronizar as métricas, convertemos para grafo não-direcionado. Análises de mão única, contramão ou sentido obrigatório exigem o grafo direcionado original.
- **Peso das arestas ignorado.** As distâncias reais das ruas (atributo `length` do OSMnx) não foram usadas como peso nas centralidades. Métricas ponderadas (`weight="length"`) poderiam revelar centralidades diferentes que consideram o comprimento real das vias.
- **Sem dados de tráfego ou capacidade.** A análise é puramente topológica. Volume de veículos, velocidades médias, horários de pico e capacidade das vias não foram modelados.
- **Snapshot temporal.** Os dados refletem o OpenStreetMap no momento do download. Mudanças urbanas posteriores (novas ruas, fechamentos) não estão refletidas.

### Aplicações práticas

A metodologia desenvolvida poderia subsidiar:
- **Planejamento de mobilidade urbana:** priorizar manutenção e investimento em nós de alta betweenness.
- **Gestão de risco e contingência:** mapear zonas vulneráveis a interdições e planejar rotas alternativas antes que sejam necessárias.
- **Análise comparativa entre bairros:** replicar a metodologia em outras regiões para comparações estruturais quantitativas.
- **Resposta a emergências:** identificar previamente os bridge nodes de cada região para priorizar liberação em caso de bloqueio.

### Reflexão final

O trabalho confirma que **grafos não são apenas uma abstração matemática — são uma ferramenta poderosa para interpretar e quantificar sistemas urbanos reais**. Mais do que isso, este projeto revelou um insight metodológico importante: **em redes planares saturadas (como malhas viárias), o grau perde força como métrica de importância**. Cruzamentos com o mesmo número de conexões podem ter papéis estruturais completamente diferentes — e só métricas globais como betweenness e core number conseguem distingui-los. A análise transforma intuições subjetivas ("essa interseção parece importante") em afirmações objetivas e mensuráveis ("este nó tem betweenness 10× maior que o décimo do ranking e é o único caminho entre duas comunidades topológicas").